# Практика · Дифузійні моделі

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає **девʼять** дифузійних моделей (три конфігурації × три зерна) і
> ще одного суддю-класифікатора, а потім відбирає з них тисячі зображень.
> Заміряно на чотирьох ядрах без відеокарти, в один потік, на завантаженій
> машині: **близько пʼяти з половиною хвилин**. Останній рядок зошита друкує
> фактичний час твого прогону.
> Найдовше йде не навчання, а **відбір**: один запуск на 200 кроків — це 200
> проходів мережею.

Тема стверджує вісім речей. Тут кожна перетворюється на число.

1. **Пряма дифузія має закриту формулу.** Порівняємо чесний ланцюг із 50 кроків
   і стрибок однією формулою — і поставимо `assert`.
2. **Книжкові `betas` при 200 кроках не доводять до чистого шуму.** Порахуємо,
   яке середнє має кінець ланцюга замість нуля.
3. **Суддя й FID.** Побудуємо власну метрику: відстань Фреше на ознаках
   класифікатора фігур.
4. **⚠️ Калібрування метрики — головний розділ зошита.** Проженемо метрику через
   вісім завідомо поганих вибірок. Одна з двох версій метрики провалиться.
5. **Крива втрати дифузії щось означає** — на відміну від GAN. Перевіримо
   кореляцією між втратою й якістю.
6. **Скільки кроків треба.** Відповімо вже каліброваною метрикою.
7. **DDIM проти DDPM** і **шум проти зображення**: дві пари, у яких різниця
   може виявитись меншою за розкид.
8. **Умовна генерація**: додамо клас на вхід і подивимось, чи слухається модель.

**Мережа не потрібна:** усі зображення ми малюємо формулами.

In [ ]:
import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import linalg

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)

# один потік, а не чотири. Мережі тут крихітні, і чотири потоки більше часу
# домовляються між собою, ніж рахують. Друга причина важливіша за швидкість:
# під кількома потоками float-суми йдуть в іншому порядку, і числа пливуть.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Датасет: ті самі шість фігур

Наскрізний приклад блоку 8 — шість фігур 28×28 із теми «Самонаглядове навчання»:
коло, квадрат, ромб, кільце, хрест, трикутник. Центр і радіус випадкові, шум
σ = 0.06. Генератор беремо дослівно, щоб числа сходилися з сусідніми темами.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.06, center=None, radius=None):
    """Малює одну фігуру як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    if center is None:
        center_y = size / 2 + rng.integers(-jitter, jitter + 1)
        center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    else:
        center_y, center_x = center
    if radius is None:
        radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    if noise > 0:
        image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    """Повертає (count, 1, 28, 28) і (count,). Класів шість, порівну."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6                       # рівно по шостій частині кожного класу
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
x_train, y_train = make_dataset(1200, rng)   # на цьому вчаться і суддя, і дифузія
x_ref, y_ref = make_dataset(1200, rng)       # еталонна купа для FID
x_test, y_test = make_dataset(600, rng)      # перевірка судді

print("навчальний набір:", tuple(x_train.shape))
print("еталонний набір :", tuple(x_ref.shape))
print("діапазон значень: %.3f … %.3f" % (x_train.min(), x_train.max()))

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 6, figsize=(9, 1.9))
for kind in range(6):
    axes[kind].imshow(x_train[kind, 0], cmap="gray", vmin=0, vmax=1)
    axes[kind].set_title(SHAPE_NAMES[kind], fontsize=9)
    axes[kind].axis("off")
plt.tight_layout()
plt.show()
print("шість класів, по 200 прикладів кожного")

## 2 · Пряма дифузія і її закрита формула

Пряма дифузія псує зображення за відомим рецептом:

```
x_t = sqrt(1 - beta_t) * x_{t-1} + sqrt(beta_t) * eps
```

Головне твердження лекції — що весь ланцюг згортається в один стрибок:

```
x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
```

де `alpha_bar_t` — добуток усіх `(1 - beta)` до кроку `t`. Перевіримо це не
на словах: прокрутимо 50 кроків чесно й порівняємо з формулою.

In [ ]:
T_STEPS = 200


def linear_schedule(n_steps, first=1e-4, last=0.02):
    """Книжковий розклад із першої статті про DDPM."""
    return torch.linspace(first, last, n_steps)


class Schedule:
    """Розклад шуму плюс усе, що з нього рахується один раз наперед."""

    def __init__(self, betas):
        self.betas = betas
        self.alphas = 1.0 - betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.n_steps = len(betas)


book = Schedule(linear_schedule(T_STEPS))

print("крок    alpha_bar   частка сигналу   частка шуму")
print("-" * 50)
for step in (10, 50, 100, 199):
    ab = float(book.alpha_bar[step])
    print("%4d   %9.6f   %12.4f   %11.4f" % (step, ab, ab ** 0.5, (1 - ab) ** 0.5))

In [ ]:
# Чесний ланцюг проти стрибка. Обидва рахуємо на тих самих 600 зображеннях,
# переведених у діапазон [-1, 1] — саме в ньому працює вся дифузія.
noise_gen = torch.Generator().manual_seed(0)
x0 = x_train[:600] * 2 - 1

x_chain = x0.clone()
for step in range(50):                       # 50 кроків, по одному, як у визначенні
    beta = book.betas[step]
    x_chain = (1 - beta).sqrt() * x_chain \
        + beta.sqrt() * torch.randn(x_chain.shape, generator=noise_gen)

ab_49 = book.alpha_bar[49]
x_jump = ab_49.sqrt() * x0 \
    + (1 - ab_49).sqrt() * torch.randn(x0.shape, generator=noise_gen)

# міряємо те саме: наскільки великий шум залишився поверх приглушеного оригіналу
sigma_chain = float((x_chain - ab_49.sqrt() * x0).std())
sigma_jump = float((x_jump - ab_49.sqrt() * x0).std())
sigma_theory = float((1 - ab_49).sqrt())

print("ітеративно 50 кроків : sigma = %.4f" % sigma_chain)
print("одна формула         : sigma = %.4f" % sigma_jump)
print("теорія sqrt(1-ᾱ₄₉)   : sigma = %.4f" % sigma_theory)

assert abs(sigma_chain - sigma_theory) < 0.01, "ланцюг розійшовся з формулою!"
assert abs(sigma_jump - sigma_theory) < 0.01, "стрибок розійшовся з формулою!"
print("✅ збігається — отже, навчати можна стрибком, а не ланцюгом")

## 3 · Розклад шуму: чи доводить ланцюг до чистого шуму

Тепер найважливіша перевірка прямого процесу, яку майже ніде не роблять.
Відбір починається з чистого шуму `N(0, 1)`. Отже, кінець ланцюга **мусить бути
схожим на `N(0, 1)`** — інакше на вході в модель опиниться те, чого вона в
навчанні не бачила.

Порахуємо для трьох розкладів, що насправді виходить на останньому кроці:
середнє й розкид справжніх `x_T`.

In [ ]:
def cosine_schedule(n_steps, s=0.008):
    """Косинусний розклад: beta задають не числами, а формою кривої alpha_bar."""
    positions = torch.arange(n_steps + 1, dtype=torch.float64) / n_steps
    f = torch.cos((positions + s) / (1 + s) * np.pi / 2) ** 2
    alpha_bar = f / f[0]
    betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
    return betas.clamp(1e-4, 0.999).float()


SCHEDULES = {
    "книжковий 0.0001…0.02": Schedule(linear_schedule(T_STEPS)),
    # ті самі числа, домножені на 1000/T: стільки шуму, скільки в статті за 1000 кроків
    "стиснутий 0.0005…0.1": Schedule(linear_schedule(T_STEPS, 5e-4, 0.1)),
    "косинусний": Schedule(cosine_schedule(T_STEPS)),
}

print("середнє яскравості справжніх фігур у [-1, 1]: %.4f" % float(x0.mean()))
print()
print("%-22s %9s %11s %11s %10s" % ("розклад", "сума β", "√ᾱ у кінці", "серед. x_T", "розкид x_T"))
print("-" * 68)
for name, sched in SCHEDULES.items():
    ab_last = sched.alpha_bar[-1]
    x_last = ab_last.sqrt() * x0 \
        + (1 - ab_last).sqrt() * torch.randn(x0.shape, generator=noise_gen)
    print("%-22s %9.3f %11.4f %+11.4f %10.4f"
          % (name, float(sched.betas.sum()), float(ab_last.sqrt()),
             float(x_last.mean()), float(x_last.std())))

Ось і діагноз. Книжкові числа `0.0001…0.02` написані для **тисячі** кроків: сума
`beta` там дорівнює приблизно 10. Узявши ті самі числа на 200 кроків, ми лишили
від потрібного шуму пʼяту частину — і кінець ланцюга має середнє **не нуль**.

Далі робочим розкладом беремо **стиснутий лінійний**. Косинусний лишаємо для
чесного порівняння в розділі 8: обидва доходять до нуля, але різною формою.

In [ ]:
WORK = SCHEDULES["стиснутий 0.0005…0.1"]
COSINE = SCHEDULES["косинусний"]

print("робочий розклад: β від %.4f до %.4f, кроків %d"
      % (WORK.betas[0], WORK.betas[-1], WORK.n_steps))
print("ᾱ на кроці 100: стиснутий %.4f, косинусний %.4f"
      % (WORK.alpha_bar[100], COSINE.alpha_bar[100]))

## 4 · Суддя: класифікатор фігур, з якого ми зробимо метрику

Генерацію не можна оцінити на око там, де кожне число мусить друкувати зошит.
Тому блок 8 уводить **суддю** — окремий класифікатор шести фігур, навчений на
справжніх даних. З нього ми візьмемо дві речі:

- **ознаки** (передостанній шар, 64 числа) — простір, у якому рахуватимемо FID;
- **відповідь** — щоб порахувати, скільки з шести класів модель узагалі малює.

⚠️ Одразу застереження, яке треба тримати в голові всю тему: наш FID рахується
на ознаках **власного судді**, а не Inception. Він годиться, щоб порівнювати наші
моделі між собою, і **не годиться**, щоб цитувати поруч із числами зі статей.

In [ ]:
class Judge(nn.Module):
    """Класифікатор фігур. Передостанній шар — 64 числа, це й буде простір FID."""

    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
        )
        self.embed = nn.Linear(288, 64)     # вузьке місце: 288 → 64
        self.head = nn.Linear(64, 6)

    def features(self, x):
        return self.embed(self.conv(x))

    def forward(self, x):
        return self.head(F.relu(self.features(x)))


torch.manual_seed(0)
judge = Judge()
judge_opt = torch.optim.AdamW(judge.parameters(), lr=3e-3)
for epoch in range(15):
    order = torch.randperm(len(x_train))
    for start in range(0, len(x_train), 128):
        idx = order[start:start + 128]
        loss = F.cross_entropy(judge(x_train[idx]), y_train[idx])
        judge_opt.zero_grad(); loss.backward(); judge_opt.step()
judge.eval()

with torch.no_grad():
    judge_accuracy = float((judge(x_test).argmax(1) == y_test).float().mean())
print("точність судді на справжніх фігурах: %.4f" % judge_accuracy)
print("тобто задача для нього майже тривіальна — і будь-яка його")
print("невпевненість на згенерованому щось означає")

## 5 · Дві версії FID — і чому їх дві

FID — це відстань Фреше між двома гаусіанами, підігнаними до хмар ознак:

```
FID = |mu_A - mu_B|^2 + tr(Cov_A + Cov_B - 2*sqrt(Cov_A @ Cov_B))
```

Словами: **наскільки далеко один від одного центри двох хмар і наскільки різна
їхня форма.** Окремій картинці FID не ставлять — він має сенс лише для набору.

Рахуватимемо його у двох різних просторах ознак, і це не примха:

- **ознаки судді (64 числа)** — вихід шару, який навчений розрізняти фігури;
- **«слабкі» ознаки (32 числа)** — карти згорток, усереднені по простору.
  Просторове усереднення викидає форму, лишаючи щось на кшталт «скільки
  чорнила де». Саме така версія метрики і є класичною пасткою.

In [ ]:
def frechet_distance(features_a, features_b):
    """Відстань Фреше між двома хмарами ознак."""
    a = np.asarray(features_a, dtype=np.float64)
    b = np.asarray(features_b, dtype=np.float64)
    mu_a, mu_b = a.mean(0), b.mean(0)
    # крихітний гребінь на діагоналі: без нього вироджені вибірки (усе чорне)
    # дають сингулярну матрицю, і корінь із неї не рахується
    ridge = 1e-6 * np.eye(a.shape[1])
    cov_a = np.cov(a, rowvar=False) + ridge
    cov_b = np.cov(b, rowvar=False) + ridge
    diff = mu_a - mu_b
    covmean, _ = linalg.sqrtm(cov_a @ cov_b, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(cov_a) + np.trace(cov_b) - 2 * np.trace(covmean))


@torch.no_grad()
def features_strong(images):
    """64 числа з передостаннього шару судді — тут форма фігури ще жива."""
    return judge.features(images).numpy()


@torch.no_grad()
def features_weak(images):
    """32 числа: карти згорток, усереднені по простору. Форму викинуто."""
    return judge.conv[:6](images).mean(dim=(2, 3)).numpy()


@torch.no_grad()
def class_shares(images):
    """Яку частку вибірки суддя відніс до кожного з шести класів."""
    predicted = judge(images).argmax(1).numpy()
    return np.bincount(predicted, minlength=6) / len(images)


N_EVAL = 200                                   # стільки зображень в одній оцінці
real_strong = features_strong(x_ref[:N_EVAL])
real_weak = features_weak(x_ref[:N_EVAL])


def score(images):
    """Пара чисел, якою ми міряємо генерацію, плюс розкладка по класах."""
    shares = class_shares(images)
    return (frechet_distance(real_strong, features_strong(images)),
            frechet_distance(real_weak, features_weak(images)),
            int((shares >= 0.05).sum()), shares)


base_strong, base_weak, base_classes, _ = score(x_ref[N_EVAL:2 * N_EVAL])
print("базова лінія — справжні фігури проти інших справжніх фігур:")
print("  FID на ознаках судді : %.4f" % base_strong)
print("  FID на слабких ознаках: %.4f" % base_weak)
print("  класів із часткою ≥ 5 %%: %d" % base_classes)
print()
print("це не нуль, і не мусить бути нулем: 200 зображень — скінченна вибірка.")
print("Друкувати цю лінію поруч із будь-яким FID обовʼязково.")

## 6 · ⚠️ Головний розділ: метрику спершу калібрують

Ось правило, заради якого написано весь цей блок курсу.

**Перш ніж вірити метриці генерації, прожени її через завідомо погані вибірки.**
Ми точно знаємо, що з них гірше й що краще: «лише кола» — це повний колапс мод,
один клас із шести. «Розмита каша» — це втрата форми. «Усе біле» — це взагалі не
зображення. Метрика, яка ставить хоч щось із цього **краще** за робочу модель,
непридатна, і всі числа, поміряні нею, доводиться викидати.

Складемо вісім таких наборів і подивимось, що скажуть обидві версії FID.

In [ ]:
def only_kinds(kinds, count, rng):
    """Вибірка лише з перелічених класів — рукотворний колапс мод."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    for i in range(count):
        images[i, 0] = draw_shape(kinds[i % len(kinds)], rng)
    return torch.from_numpy(images)


def blur_batch(images, passes=3):
    """Тричі розмиває вікном 5×5 — форма зникає, «чорнило» лишається."""
    data = images.numpy().copy()
    out = np.zeros_like(data)
    for i in range(len(data)):
        img = data[i, 0]
        for _ in range(passes):
            padded = np.pad(img, 2, mode="edge")
            img = sum(padded[a:a + 28, b:b + 28] for a in range(5) for b in range(5)) / 25.0
        out[i, 0] = img
    return torch.from_numpy(out)


bad_rng = np.random.default_rng(7)
held_out = x_ref[N_EVAL:2 * N_EVAL]

CALIBRATION = {
    "справжні (інша вибірка)": held_out,
    "лише кола": only_kinds([0], N_EVAL, bad_rng),
    "кола + квадрати": only_kinds([0, 1], N_EVAL, bad_rng),
    "розмита каша": blur_batch(held_out),
    "середнє всіх фігур": held_out.mean(dim=0, keepdim=True).repeat(N_EVAL, 1, 1, 1),
    "усе чорне": torch.zeros(N_EVAL, 1, 28, 28),
    "чистий шум": torch.from_numpy(bad_rng.random((N_EVAL, 1, 28, 28)).astype(np.float32)),
    "усе біле": torch.ones(N_EVAL, 1, 28, 28),
}

calibration_rows = {}
print("%-24s %12s %12s %8s" % ("що подаємо", "FID судді", "FID слабкий", "класів"))
print("-" * 60)
for name, batch in CALIBRATION.items():
    strong, weak, classes, shares = score(batch)
    calibration_rows[name] = (strong, weak, classes, shares)
    print("%-24s %12.4f %12.4f %8d" % (name, strong, weak, classes))

In [ ]:
# Проглянемо очима те, що поставили метриці на калібрування
figure, axes = plt.subplots(1, 8, figsize=(12, 1.9))
for ax, (name, batch) in zip(axes, CALIBRATION.items()):
    ax.imshow(batch[0, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(name.split(" (")[0], fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("зліва направо — від найкращого набору до найгіршого")

In [ ]:
# Формальна перевірка: чи росте FID разом із погіршенням набору?
order_by_badness = ["справжні (інша вибірка)", "лише кола", "кола + квадрати",
                    "розмита каша", "середнє всіх фігур", "усе чорне",
                    "чистий шум", "усе біле"]

strong_series = [calibration_rows[n][0] for n in order_by_badness]
weak_series = [calibration_rows[n][1] for n in order_by_badness]

def monotone(series):
    return all(series[i] < series[i + 1] for i in range(len(series) - 1))

print("FID на ознаках судді росте разом із погіршенням: ", monotone(strong_series))
print("FID на слабких ознаках росте разом із погіршенням:", monotone(weak_series))
print()
print("Ключове порівняння — колапс мод проти справжніх даних:")
print("  ознаки судді : справжні %.4f  проти  «лише кола» %.4f  → у %.0f разів гірше"
      % (strong_series[0], strong_series[1], strong_series[1] / strong_series[0]))
print("  слабкі ознаки: справжні %.4f  проти  «лише кола» %.4f  → у %.0f разів гірше"
      % (weak_series[0], weak_series[1], weak_series[1] / weak_series[0]))

Обидві версії впорядковують набори правильно — але шкала в них геть різна, і саме
в шкалі вся справа. Для слабких ознак повний колапс мод стоїть **поруч** із базовою
лінією: різниця там у сотих. Це означає, що робоча модель із будь-якою помітною
похибкою легко опиниться **гірше за повний колапс** — і метрика тихо перекине
висновок.

І ще одне, чого не робить жодна з двох версій FID поодинці: **відрізнити «лише
кола» від «кола + квадрати»**. Числа майже однакові. Один клас із шести й два
класи з шести — це різні хвороби, а FID їх не розводить.

**Звідси правило теми: метрика генерації — це пара.** FID на ознаках судді
показує, наскільки хмара згенерованого схожа на хмару справжнього; покриття
класів показує, чи не забула модель половину даних. Далі скрізь друкуємо обидва
числа й базову лінію поруч.

## 7 · Мережа знешумлення

Маленький U-Net: 28×28 → 14×14 → 7×7 і назад, зі скіп-зʼєднаннями. Номер кроку
подається як набір синусів і косинусів різної частоти — так само, як позиції
токенів у трансформері.

Модель бере ще й **номер класу**. У десяти відсотках прикладів замість класу
підставляється позначка «невідомо» — і тоді одна й та сама мережа вміє і малювати
на замовлення, і малювати будь-що. Це найдешевший спосіб мати обидва режими, і
саме на ньому тримається керування текстом у великих моделях.

In [ ]:
def time_embedding(t, dim=32):
    """Номер кроку → набір синусів і косинусів різної частоти."""
    half = dim // 2
    freqs = torch.exp(-np.log(1000.0) * torch.arange(half, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class DenoiseNet(nn.Module):
    """Крихітний U-Net: 28 → 14 → 7 → 14 → 28, час і клас — у кожному блоці."""

    def __init__(self, base=8, n_classes=6, tdim=32):
        super().__init__()
        self.tdim = tdim
        self.time_mlp = nn.Sequential(nn.Linear(tdim, tdim), nn.ReLU(), nn.Linear(tdim, tdim))
        # класів шість плюс сьома позначка «невідомо»
        self.class_emb = nn.Embedding(n_classes + 1, tdim)
        b = base
        self.stem = nn.Conv2d(1, b, 3, padding=1)            # 1→8   на 28×28
        self.enc = nn.Conv2d(b, b * 2, 3, padding=1)         # 8→16  на 14×14
        self.mid1 = nn.Conv2d(b * 2, b * 4, 3, padding=1)    # 16→32 на 7×7
        self.mid2 = nn.Conv2d(b * 4, b * 4, 3, padding=1)    # 32→32 на 7×7
        # 1×1 у декодері: згортка 3×3 на 28×28 коштувала б утричі більше за все інше
        self.dec1 = nn.Conv2d(b * 6, b * 2, 1)               # 48→16 на 14×14
        self.dec2 = nn.Conv2d(b * 3, b, 1)                   # 24→8  на 28×28
        self.out = nn.Conv2d(b, 1, 3, padding=1)             # 8→1   на 28×28
        self.t1 = nn.Linear(tdim, b * 2)
        self.t2 = nn.Linear(tdim, b * 4)
        self.t3 = nn.Linear(tdim, b * 4)
        self.t4 = nn.Linear(tdim, b * 2)

    def forward(self, x, t, y):
        cond = self.time_mlp(time_embedding(t, self.tdim)) + self.class_emb(y)
        e1 = F.relu(self.stem(x))                                     # 8 × 28 × 28
        e2 = F.relu(self.enc(F.avg_pool2d(e1, 2)) + self.t1(cond)[:, :, None, None])
        m = F.relu(self.mid1(F.avg_pool2d(e2, 2)) + self.t2(cond)[:, :, None, None])
        m = F.relu(self.mid2(m) + self.t3(cond)[:, :, None, None])
        u1 = torch.cat([F.interpolate(m, size=14, mode="nearest"), e2], 1)
        d1 = F.relu(self.dec1(u1) + self.t4(cond)[:, :, None, None])
        u2 = torch.cat([F.interpolate(d1, size=28, mode="nearest"), e1], 1)
        return self.out(F.relu(self.dec2(u2)))


NO_CLASS = 6                                   # позначка «клас невідомий»
print("параметрів у мережі:", sum(p.numel() for p in DenoiseNet().parameters()))

## 8 · Навчання і зворотний прохід

Навчання — це буквально сім рядків: кинути жереб на `t`, стрибнути на цей крок
формулою, спитати мережу про шум, порівняти. Ніякого ланцюга, ніякого змагання.

Зворотний прохід складніший, і в ньому є пастка, яка коштувала мені години.
Здогад про чисте зображення обрізають до `[-1, 1]`. Після обрізання **не можна**
користуватись формулою переходу у вигляді «через `x0` і `eps`»: два доданки
перестають узгоджуватись між собою, і ланцюг розходиться — розкид `x` сягав 153
замість одиниці. Правильна форма — «через `x0` і поточне `x_t`», вона нижче.

In [ ]:
def train_diffusion(images, labels, sched, seed=0, epochs=22, target="noise",
                    lr=6e-3, batch=128, drop_prob=0.1, checkpoints=()):
    """Навчає одну дифузійну модель. Повертає модель, історію втрати й зрізи."""
    torch.manual_seed(seed)
    model = DenoiseNet()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    n = len(images)
    history, snapshots = [], {}
    for epoch in range(1, epochs + 1):
        order = torch.randperm(n)
        total = 0.0
        for start in range(0, n, batch):
            idx = order[start:start + batch]
            x_clean = images[idx] * 2 - 1                 # дифузія живе в [-1, 1]
            t = torch.randint(0, sched.n_steps, (len(idx),))
            noise = torch.randn_like(x_clean)
            ab = sched.alpha_bar[t][:, None, None, None]
            x_noisy = ab.sqrt() * x_clean + (1 - ab).sqrt() * noise

            y = labels[idx].clone()
            # у частині прикладів ховаємо клас — так модель учиться й без умови
            y[torch.rand(len(idx)) < drop_prob] = NO_CLASS

            prediction = model(x_noisy, t, y)
            loss = F.mse_loss(prediction, noise if target == "noise" else x_clean)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        history.append(total / n)
        if epoch in checkpoints:
            snapshots[epoch] = copy.deepcopy(model).eval()
    model.eval()
    return model, history, snapshots


@torch.no_grad()
def sample(model, sched, n_samples, steps=None, target="noise", seed=0,
           ddim=False, labels=None, chunk=100):
    """Зворотний прохід: із чистого шуму в зображення.

    Рахуємо порціями по 100: на великому батчі проміжні тензори вилітають
    із кешу процесора, і та сама робота йде вдвічі довше.
    """
    if n_samples > chunk:
        parts = []
        for start in range(0, n_samples, chunk):
            size = min(chunk, n_samples - start)
            part_labels = None if labels is None else labels[start:start + size]
            parts.append(sample(model, sched, size, steps=steps, target=target,
                                seed=seed * 1000 + start, ddim=ddim,
                                labels=part_labels, chunk=chunk))
        return torch.cat(parts, 0)

    gen = torch.Generator().manual_seed(seed)
    x = torch.randn(n_samples, 1, 28, 28, generator=gen)
    if labels is None:
        labels = torch.full((n_samples,), NO_CLASS, dtype=torch.long)

    last = sched.n_steps - 1
    if steps is None or steps >= sched.n_steps:
        times = list(range(last, -1, -1))
    else:
        times = list(np.linspace(last, 0, steps).round().astype(int))

    for i, t in enumerate(times):
        t_batch = torch.full((n_samples,), int(t), dtype=torch.long)
        prediction = model(x, t_batch, labels)
        ab_t = sched.alpha_bar[t]
        if target == "noise":
            x_start = (x - (1 - ab_t).sqrt() * prediction) / ab_t.sqrt()
        else:
            x_start = prediction
        x_start = x_start.clamp(-1, 1)          # яскравіше за біле не буває

        t_prev = times[i + 1] if i + 1 < len(times) else -1
        if t_prev < 0:
            x = x_start
            break
        ab_prev = sched.alpha_bar[t_prev]
        alpha_seg = ab_t / ab_prev              # добуток alpha на пропущеному відрізку
        if ddim:
            # детерміновано: свіжого шуму не додаємо зовсім
            eps = (x - ab_t.sqrt() * x_start) / (1 - ab_t).sqrt()
            x = ab_prev.sqrt() * x_start + (1 - ab_prev).sqrt() * eps
        else:
            var = (1 - ab_prev) / (1 - ab_t) * (1 - alpha_seg)
            mean = (ab_prev.sqrt() * (1 - alpha_seg) / (1 - ab_t)) * x_start \
                 + (alpha_seg.sqrt() * (1 - ab_prev) / (1 - ab_t)) * x
            x = mean + var.clamp(min=0).sqrt() * torch.randn(x.shape, generator=gen)
    return ((x + 1) / 2).clamp(0, 1)


print("навчання і відбір визначені")

## 9 · Навчаємо девʼять моделей

Три конфігурації, по три зерна на кожну:

| конфігурація | розклад | що передбачає |
|---|---|---|
| `шум` | стиснутий лінійний | підмішаний шум |
| `косинус` | косинусний | підмішаний шум |
| `зображення` | стиснутий лінійний | чисте зображення |

Це найдовша клітинка зошита. У першої конфігурації ми ще й зберігаємо зрізи
моделі після 4, 10 і 22 епох — на них перевірятимемо, чи означає щось крива втрати.

In [ ]:
EPOCHS = 22
CHECKPOINTS = (4, 10, EPOCHS)
CONFIGS = {
    "шум": dict(sched=WORK, target="noise", checkpoints=CHECKPOINTS),
    "косинус": dict(sched=COSINE, target="noise", checkpoints=()),
    "зображення": dict(sched=WORK, target="image", checkpoints=()),
}

models, losses, snaps = {}, {}, {}
train_started = time.perf_counter()
for name, cfg in CONFIGS.items():
    for seed in range(3):
        started = time.perf_counter()
        model, history, snapshots = train_diffusion(
            x_train, y_train, cfg["sched"], seed=seed, epochs=EPOCHS,
            target=cfg["target"], checkpoints=cfg["checkpoints"])
        models[(name, seed)] = model
        losses[(name, seed)] = history
        snaps[(name, seed)] = snapshots
        print("%-11s зерно %d: %5.1f с, втрата %.4f → %.4f"
              % (name, seed, time.perf_counter() - started, history[0], history[-1]))
print()
print("усе навчання: %.1f с" % (time.perf_counter() - train_started))

## 10 · Чи означає щось крива втрати

У GAN втрата генератора нічого не каже про якість: вона міряється проти
дискримінатора, який сам змінюється. У дифузії втрата одна й має мінімум. Але
«має мінімум» ще не означає «падає разом із якістю картинок» — це треба перевірити.

Візьмемо збережені зрізи після 4, 10 і 22 епох, відберемо з кожного по 120
зображень і поставимо поруч втрату й FID.

In [ ]:
loss_vs_fid = []
for epoch in CHECKPOINTS:
    fids, coverage = [], []
    for seed in range(3):
        generated = sample(snaps[("шум", seed)][epoch], WORK, 120, steps=20,
                           ddim=True, seed=seed)
        strong, _, classes, _ = score(generated)
        fids.append(strong); coverage.append(classes)
    mean_loss = float(np.mean([losses[("шум", seed)][epoch - 1] for seed in range(3)]))
    loss_vs_fid.append((epoch, mean_loss, float(np.mean(fids)), float(np.std(fids)),
                        coverage))
    print("епоха %2d: втрата %.4f   FID %8.2f ± %6.2f   класів %s"
          % (epoch, mean_loss, np.mean(fids), np.std(fids), coverage))

losses_only = [row[1] for row in loss_vs_fid]
fids_only = [row[2] for row in loss_vs_fid]
correlation = float(np.corrcoef(losses_only, fids_only)[0, 1])
print()
print("кореляція «втрата ↔ FID» на трьох контрольних точках: %+.4f" % correlation)

## 11 · Скільки кроків насправді треба

Тепер головне питання практики. Дифузія дорога саме тому, що одне зображення
коштує десятки проходів мережею. Порахуємо, як якість залежить від кількості
кроків — і зробимо це **вже каліброваною парою метрик**, а не однією.

Дві схеми відбору:

- **DDPM** — на кожному кроці додає свіжий шум `sigma * z`;
- **DDIM** — не додає нічого. Той самий здогад про `x0`, але детермінований
  перехід. Саме через це DDIM переживає пропуск кроків.

Обидві версії FID рахуємо **за один прохід**: другий стовпчик нічого не коштує,
бо ознаки вже витягнуті.

In [ ]:
STEP_GRID = [2, 5, 20, 200]
sweep = {}
sweep_started = time.perf_counter()

print("%-6s %6s %11s %9s %8s   %s"
      % ("схема", "кроків", "FID", "розкид", "класів", "по зернах"))
print("-" * 74)
for scheme, is_ddim in (("DDIM", True), ("DDPM", False)):
    grid = STEP_GRID if is_ddim else [5, 200]
    for steps in grid:
        strong_all, weak_all, coverage, share_rows = [], [], [], []
        for seed in range(3):
            generated = sample(models[("шум", seed)], WORK, N_EVAL, steps=steps,
                               ddim=is_ddim, seed=seed)
            strong, weak, classes, shares = score(generated)
            strong_all.append(strong); weak_all.append(weak); coverage.append(classes)
            share_rows.append(shares)
        sweep[(scheme, steps)] = dict(fid=strong_all, weak=weak_all, classes=coverage,
                                      shares=np.mean(share_rows, axis=0))
        print("%-6s %6d %11.2f %9.2f %8s   %s"
              % (scheme, steps, np.mean(strong_all), np.std(strong_all), coverage,
                 [round(v, 1) for v in strong_all]))
print()
print("відбір зайняв %.1f с" % (time.perf_counter() - sweep_started))
print("базова лінія (справжні проти справжніх): %.4f" % base_strong)

In [ ]:
# Той самий відбір очима зламаної метрики. Нічого не переробляємо — просто
# дивимось на стовпчик, який уже порахований разом із першим.
print("%-6s %6s %14s %14s %8s" % ("схема", "кроків", "FID судді", "FID слабкий", "класів"))
print("-" * 58)
for steps in STEP_GRID:
    row = sweep[("DDIM", steps)]
    print("%-6s %6d %14.4f %14.4f %8.1f"
          % ("DDIM", steps, np.mean(row["fid"]), np.mean(row["weak"]),
             np.mean(row["classes"])))

best_strong = min(STEP_GRID, key=lambda s: np.mean(sweep[("DDIM", s)]["fid"]))
best_weak = min(STEP_GRID, key=lambda s: np.mean(sweep[("DDIM", s)]["weak"]))
print()
print("найкраще за FID судді  : %d кроків" % best_strong)
print("найкраще за слабким FID: %d кроків" % best_weak)
print("покриття класів там і там: %.1f і %.1f"
      % (np.mean(sweep[("DDIM", best_strong)]["classes"]),
         np.mean(sweep[("DDIM", best_weak)]["classes"])))

## 12 · Три пари, у яких різниця може виявитись меншою за розкид

Далі порівняння, і в кожному ми дивимось не на середні, а на **купи по зернах**.
Правило курсу: різниця, менша за розкид, не є різницею; непересічні купи при
трьох зернах — сильніший аргумент, ніж різниця середніх.

In [ ]:
COMPARE_STEPS = 20    # стільки, скільки виявилось досить у розділі 11
comparison = {"шум": dict(fid=sweep[("DDIM", COMPARE_STEPS)]["fid"],
                          classes=sweep[("DDIM", COMPARE_STEPS)]["classes"])}
for name in ("косинус", "зображення"):
    sched = CONFIGS[name]["sched"]
    target = CONFIGS[name]["target"]
    fids, coverage = [], []
    for seed in range(3):
        generated = sample(models[(name, seed)], sched, N_EVAL, steps=COMPARE_STEPS,
                           target=target, ddim=True, seed=seed)
        strong, weak, classes, shares = score(generated)
        fids.append(strong); coverage.append(classes)
    comparison[name] = dict(fid=fids, classes=coverage)
for name in ("шум", "косинус", "зображення"):
    fids = comparison[name]["fid"]
    coverage = comparison[name]["classes"]
    print("%-11s FID %8.2f ± %6.2f   по зернах %s   класів %s"
          % (name, np.mean(fids), np.std(fids), [round(f, 1) for f in fids], coverage))

In [ ]:
def piles_overlap(a, b):
    '''Чи перетинаються дві купи значень. Купи, що не перетинаються, — доказ.'''
    return not (max(a) < min(b) or max(b) < min(a))


pairs = [("шум", "зображення"), ("шум", "косинус")]
for left, right in pairs:
    a, b = comparison[left]["fid"], comparison[right]["fid"]
    overlap = piles_overlap(a, b)
    print("%-11s проти %-11s: середні %.1f і %.1f, купи %s"
          % (left, right, np.mean(a), np.mean(b),
             "ПЕРЕТИНАЮТЬСЯ — різниці не доведено" if overlap else "не перетинаються"))

# DDIM проти DDPM на тій самій кількості кроків — теж пара куп
for steps in (5, 200):
    a = sweep[("DDIM", steps)]["fid"]
    b = sweep[("DDPM", steps)]["fid"]
    print("%3d кроків: DDIM %8.1f проти DDPM %8.1f, купи %s"
          % (steps, np.mean(a), np.mean(b),
             "ПЕРЕТИНАЮТЬСЯ" if piles_overlap(a, b) else "не перетинаються"))

### Повертаємось до калібрування: чи пройшла метрика перевірку

Тепер, коли робоча модель існує й поміряна, можна закрити питання розділу 6.
Метрика придатна тоді й тільки тоді, коли вона ставить **робочу модель краще**
за завідомо зіпсовані набори. Порівняємо прямо.

In [ ]:
model_strong = float(np.mean(sweep[("DDIM", 200)]["fid"]))
model_weak = float(np.mean(sweep[("DDIM", 200)]["weak"]))
model_classes = float(np.mean(sweep[("DDIM", 200)]["classes"]))
print("робоча модель: FID судді %.4f, FID слабкий %.4f, класів %.1f"
      % (model_strong, model_weak, model_classes))
print()
print("%-22s %11s %9s %11s %9s" % ("що подаємо", "FID судді", "у скільки",
                                   "FID слабкий", "у скільки"))
print("-" * 68)
worst_strong, worst_weak = 1e9, 1e9
for name in ("лише кола", "кола + квадрати", "розмита каша",
             "середнє всіх фігур", "усе чорне", "чистий шум", "усе біле"):
    strong, weak, classes, _ = calibration_rows[name]
    ratio_s = strong / model_strong
    ratio_w = weak / model_weak
    worst_strong = min(worst_strong, ratio_s)
    worst_weak = min(worst_weak, ratio_w)
    print("%-22s %11.4f %8.2f× %11.4f %8.2f×" % (name, strong, ratio_s, weak, ratio_w))
print()
print("найменший запас метрики судді   : %.2f×" % worst_strong)
print("найменший запас слабкої метрики : %.2f×" % worst_weak)
print()
print("Запас 1.00× означає, що метрика не відрізняє зіпсований набір")
print("від робочої моделі. Метрика з таким запасом непридатна.")
print()
print("«лише кола» — це повний колапс мод, один клас із шести.")
print("Метрика, яка ставить його краще за робочу модель, непридатна:")
print("  ознаки судді : колапс %.2f проти моделі %.2f" % (calibration_rows["лише кола"][0], model_strong))
print("  слабкі ознаки: колапс %.4f проти моделі %.4f" % (calibration_rows["лише кола"][1], model_weak))

## 13 · Умовна генерація: чи слухається модель

Модель бачила номер класу під час навчання, а в десяти відсотках прикладів —
позначку «невідомо». Отже, її можна попросити намалювати конкретну фігуру.
Перевіримо суддею, чи виходить те, що замовляли. Усі шість класів беремо
**одним відбором**: 300 зображень, у яких мітки йдуть по колу.

In [ ]:
mixed_labels = torch.arange(300) % 6          # по 50 замовлень на кожен клас
per_class_seeds = []
for seed in range(3):
    generated = sample(models[("шум", seed)], WORK, 300, steps=COMPARE_STEPS,
                       ddim=True, seed=seed, labels=mixed_labels)
    with torch.no_grad():
        predicted = judge(generated).argmax(1)
    per_class_seeds.append([
        float((predicted[mixed_labels == kind] == kind).float().mean())
        for kind in range(6)])

conditional_hits = {}
for kind in range(6):
    hits = [per_class_seeds[seed][kind] for seed in range(3)]
    conditional_hits[kind] = hits
    print("%-11s замовили → отримали: %.4f ± %.4f   по зернах %s"
          % (SHAPE_NAMES[kind], np.mean(hits), np.std(hits), [round(h, 2) for h in hits]))

base_shares = np.array(sweep[("DDIM", COMPARE_STEPS)]["shares"])
print()
print("%-11s %10s %12s %8s" % ("клас", "без умови", "на замовлення", "підйом"))
print("-" * 46)
for kind in range(6):
    hit = float(np.mean(conditional_hits[kind]))
    print("%-11s %10.3f %12.3f %8.2f×"
          % (SHAPE_NAMES[kind], base_shares[kind], hit,
             hit / max(base_shares[kind], 1e-6)))
print()
print("середнє влучання по шести класах: %.4f"
      % np.mean([np.mean(v) for v in conditional_hits.values()]))
print("рівень випадкового вгадування   : %.4f" % (1 / 6))

In [ ]:
# Та сама модель без умови: які класи вона малює, коли її не просять
unconditional_shares = base_shares
print("частки класів без умови, середнє по трьох зернах:")
for kind in range(6):
    print("  %-11s %.3f" % (SHAPE_NAMES[kind], unconditional_shares[kind]))
print("рівномірно було б по %.3f" % (1 / 6))
print("класів із часткою ≥ 5 %%: %d" % int((unconditional_shares >= 0.05).sum()))

In [ ]:
# Подивимось очима: рядок — замовлений клас, шість запусків у рядку
grid_labels = torch.arange(36) // 6
grid = sample(models[("шум", 0)], WORK, 36, steps=COMPARE_STEPS,
              ddim=True, seed=100, labels=grid_labels)
figure, axes = plt.subplots(6, 6, figsize=(6.6, 6.8))
for kind in range(6):
    for column in range(6):
        axes[kind, column].imshow(grid[kind * 6 + column, 0], cmap="gray", vmin=0, vmax=1)
        axes[kind, column].axis("off")
plt.tight_layout()
plt.show()
print("рядки згори вниз:", ", ".join(SHAPE_NAMES))

In [ ]:
print("увесь зошит: %.1f с" % (time.perf_counter() - notebook_started))

## Завдання

### 🟢 Рівень 1
Постав `T_STEPS = 1000` і побудуй книжковий розклад `linear_schedule(1000)`.
Порахуй `sqrt(alpha_bar)` на останньому кроці й середнє `x_T`. Переконайся, що
проблема з розділу 3 зникла, і поясни одним реченням чому.

### 🟡 Рівень 2
Додай до таблиці калібрування девʼятий набір: **справжні фігури, зсунуті на
5 пікселів праворуч**. Це не колапс мод і не каша — форма ціла, зміщено лише
положення. Подивись, що скажуть обидві версії FID і покриття класів, і зроби
висновок, яку саме поломку кожна з метрик бачить, а яку — ні.

### 🔴 Рівень 3
Реалізуй **classifier-free guidance**: під час відбору рахуй прогноз двічі —
з класом і з позначкою `NO_CLASS` — і бери `eps_uncond + w * (eps_cond - eps_uncond)`
при `w` від 1 до 5. Побудуй залежність «влучання в замовлений клас» і «FID» від `w`
на трьох зернах. Очікуй компроміс: більша `w` має підвищувати влучання й псувати
різноманіття.